# 4-2 47 ファイルの一括読み込み

4-1 で確認したとおり、`sales_2026-01/` の中には **47 個の Excel ファイル** が並んでいます。これを **1 つの DataFrame** にまとめましょう。

やることは:

1. **`glob`** で対象ファイルを **パターンで一気にリストアップ**
2. **`for` ループ** で 1 ファイルずつ `read_excel`
3. **`pd.concat`** で縦に積んで 1 つの DataFrame に
4. ついでに **ファイル名から都道府県コードを取り出して列にする**

ぜんぶで **5〜6 行** で済みます。

## Excel との対比

| やりたいこと | Excel + VBA | pandas |
|---|---|---|
| フォルダ内のファイル一覧を取る | `Dir` 関数 | `Path(...).glob("*.xlsx")` |
| ファイルを開く | `Workbooks.Open` (Excel 起動) | `pd.read_excel` (直接パース) |
| 全ファイルを縦に結合 | 1 行ずつ転記 | `pd.concat` (1 行) |
| ファイル名から情報を抜く | `Split` で文字分割 | `Path.stem.split("_")` |

VBA だと「ファイル開く → 行ループ → 値を別シートに転記 → 閉じる」を全ファイル繰り返す必要があり、**数十行のマクロ** になります。pandas なら **5 行** です。

## 準備

In [ ]:
# === Colab で進める人 ===
# from google.colab import drive
# drive.mount('/content/drive')
# BASE = "/content/drive/MyDrive/python-data-basics"

# === ローカルで進める人 ===
BASE = "."

DATA = f"{BASE}/data/capstone"
SALES_DIR = f"{DATA}/sales_2026-01"

import pandas as pd
from pathlib import Path

## 1. `glob` で対象ファイルを一気にリストアップ

`pathlib.Path(...).glob(パターン)` を使うと、**ワイルドカード `*` を含むパターン** にマッチするファイルが一気に取れます。

Excel でいう「**フォルダの中で名前が一致するものを全部選ぶ**」操作です。

In [ ]:
files = sorted(Path(SALES_DIR).glob("sales_2026-01_*.xlsx"))

print(f"見つかったファイル: {len(files)} 個")
files[:5]

**ポイント**:

- **`*` は「何でもいい」** という意味の記号。`sales_2026-01_*.xlsx` は「`sales_2026-01_` で始まり `.xlsx` で終わるもの全部」
- `sorted(...)` で **アルファベット順に並べる** → ファイル名にコードを入れてあるので、結果として都道府県コード順 (01→47) になる
- 取れるのは `PosixPath` オブジェクト。`.name` で `sales_2026-01_01_hokkaido.xlsx`、`.stem` で `sales_2026-01_01_hokkaido` のように扱える

## 2. `for` ループで 1 ファイルずつ読み込む

あとは 47 ファイル分 `read_excel` を回すだけです。**読み込んだ DataFrame を、後で結合するために `list` に貯めておく** のがコツです。

In [ ]:
dfs = []   # ファイルごとの DataFrame をいったんここに貯める

for f in files:
    df = pd.read_excel(f, sheet_name="売上")
    dfs.append(df)

print(f"読み込み完了。DataFrame の数: {len(dfs)}")
print(f"1 ファイル目の行数: {len(dfs[0])}")

> 💡 47 ファイル分の `read_excel` で **数秒〜十数秒** かかります。VBA で同じことをすると **数十秒〜数分** かかります（4-7 で実測比較します）。

## 3. `pd.concat` で 1 つに束ねる

`pd.concat(リスト)` で **複数の DataFrame を縦に積んで 1 つに** できます。Excel でいう「ぜんぶのシートをコピペで 1 シートにまとめる」操作。

`ignore_index=True` を付けると、結合後にインデックスを 0 から振り直してくれます（これを付けないと、ファイルごとに同じ行番号が残ってしまう）。

In [ ]:
all_sales = pd.concat(dfs, ignore_index=True)

print(f"合計行数: {len(all_sales):,}")
all_sales.head()

**47,000 行** が 1 つの DataFrame になりました。これを **Excel でやろうとすると、ファイルを 47 個開いて、シートをコピーして 1 つに貼り付けて…** という気の遠くなる作業です。

## 4. ファイル名から都道府県コードを取り出して列にする

今の `all_sales` には **「どのファイル (= どの県) の行か」が分からない** という弱点があります。3 章までで learn した `groupby` を県別にやるためには、**県を表す列が必要** です。

幸い、ファイル名に **都道府県コードが埋め込まれている** ので、そこから取り出せます。

```
sales_2026-01_13_tokyo.xlsx
          ↑   ↑
          年月  都道府県コード (この場合 13 = 東京都)
```

In [ ]:
# 練習: 1 ファイル名からコードを取り出してみる
sample = files[12]   # 13番目 = 東京
print("ファイル名:", sample.name)
print("stem    :", sample.stem)            # 拡張子なし
print("split   :", sample.stem.split("_")) # _ で分割
print("県コード :", sample.stem.split("_")[2])  # 3 番目が県コード

やり方が分かったので、**`for` ループの中で県コード列を付けながら** 読み込み直します。

In [ ]:
dfs = []

for f in files:
    df = pd.read_excel(f, sheet_name="売上")
    # ファイル名から県コードを取り出して、新しい列として全行に貼る
    df["prefecture_code"] = int(f.stem.split("_")[2])
    dfs.append(df)

all_sales = pd.concat(dfs, ignore_index=True)
all_sales.head()

右端に **`prefecture_code` 列** が増えました。`int(...)` で文字列 `"13"` を数値 `13` に変換しているのがミソ（後でマスタと結合するときに型を揃えるため）。

ここまでが、**第 4 章で一番「Python っぽい」コード** です。これさえできれば、後は 2 章・3 章で習ったことを `all_sales` に適用していくだけ。

## さっそく集計してみる — 県別売上 TOP10

せっかく束ねたので、**2-3 で習った `groupby`** をかけてみましょう。

In [ ]:
# 県コード別の売上合計 TOP10
all_sales.groupby("prefecture_code")["amount"].sum().sort_values(ascending=False).head(10)

県コード 13 (東京) が一番上に来ているはず。**でも「13」だけ表示されても直感的じゃない** ですね。県名 (`東京都`) や地域 (`関東`) を一緒に表示したい。

それを叶えるのが **マスタ結合 (`merge`)**。次の **4-3** で扱います。

## 練習問題

本文のコードを少しだけ変える形の演習です。

1. **`all_sales` の商品別売上 TOP3** を出してください。本文 cell-19 の **`prefecture_code` を `product_name` に変える** だけ。
2. **「東京都 (`prefecture_code == 13`) の取引だけ」** で `head(5)` を表示してください。2-3 で習った **boolean indexing** (`df[df["列"] == 値]`) を使う。
3. **県別の取引件数** （売上合計ではなく `len` でも `count` でもいい）TOP5 を出してください。本文 cell-19 の **`.sum()` を `.count()` に変える** だけ。

In [ ]:
# ここに書いてみてください


<details>
<summary>答えを見る</summary>

```python
# 1. 商品別 売上 TOP3
all_sales.groupby("product_name")["amount"].sum().sort_values(ascending=False).head(3)

# 2. 東京都だけ
all_sales[all_sales["prefecture_code"] == 13].head(5)

# 3. 県別 取引件数 TOP5
all_sales.groupby("prefecture_code")["amount"].count().sort_values(ascending=False).head(5)
```

</details>

## よくあるエラー

### 1. `glob` が 0 件しか返さない
→ パスが間違っているか、パターンが間違っている。`Path(SALES_DIR).exists()` で確認、`list(Path(SALES_DIR).iterdir())` で中身を見る。

### 2. `pd.concat` で `ValueError: No objects to concatenate`
→ 渡したリスト `dfs` が空。`glob` の結果が 0 件のときに起きる。`len(files)` で先に確認する。

### 3. 各ファイルの列名がバラバラだと concat 後の DataFrame が変
→ ファイルごとに列名が違うと、`concat` は **全列の和集合** を作り、足りないセルは NaN になる。本講座のサンプルデータは全ファイル同じ列名なので問題なし。

### 4. `read_excel` が遅すぎる
→ 47 ファイルでも 5〜15 秒程度かかるのは普通。それでも VBA より圧倒的に速い。**実務で気になるレベルなら `engine="calamine"` を試す**（速度 2〜5 倍、要 `pip install python-calamine`）。

### 5. ファイル名から県コードを取り出すと `"01"` のように先頭ゼロが付く
→ `int(...)` で囲むと数値の `1` になり、マスタの `prefecture_code` (整数) と結合しやすい。逆にゼロパディングを保ちたいときは `str` のまま使う。

## まとめ

- **`Path(...).glob("*.xlsx")`** でフォルダ内のファイルを一気に拾える
- **`for + read_excel`** で読み込み、**`list` に貯めて `pd.concat`** で 1 つに束ねる
- **ファイル名に埋め込まれた情報** (今回は県コード) を `stem.split("_")` で取り出して列に追加
- 結合後の DataFrame は、これまでの 2 章・3 章でやってきた **`groupby` `sort_values` などをそのまま適用** できる

47 ファイル × 1,000 行を 1 つに束ねる作業が **5 行** で終わる。VBA で書こうとすると 30〜40 行かかります。

次の **4-3** では、束ねた `all_sales` に **商品マスタ・支店マスタを `merge` で結合** して、「東京都の高級果物カテゴリの売上」みたいな分析ができる形に持っていきます。